In [ ]:
from datasets import load_from_disk

In [ ]:
from evaluate import load

In [ ]:
from evaluate import load
bertscore = load("bertscore")


In [ ]:
import json


In [ ]:
with open("generated_definitions_for_bertscore.json", 'r') as fp:
    true_defs = json.load(fp)

In [ ]:
# fnames = ["baseline_generations_lr_0.0003_bea6ecf2-2fab-443d-a096-98b12357b84e",
# "baseline_generations_lr_0.0003_ef46996f-7519-48ee-adaf-b97c4f7c72ca",
# "baseline_generations_lr_0.0003_f932d5ed-bbc5-4180-8d64-d2986455f62f"]
fnames = ["emb_gen_generations_masked_new_token_new_data_new_model_90318e95-91c0-479d-8ab9-a50a022b519a",
"emb_gen_generations_masked_new_token_new_data_new_model_a7a9fae8-1153-4b96-92ce-29fa6ab23602",
"emb_gen_generations_masked_new_token_new_data_new_model_a9d91273-5e88-468d-8a24-d9e3778f00e2"]

base = "definition_task_outputs/" 

In [ ]:
scores = {}
step = 2
for name in fnames:
    data = load_from_disk(base + name)
#     data = data.filter(lambda ex: ex['step'] == step)
    for with_prompt in [False, True]:
        if with_prompt:
            outputs = data.filter(lambda ex: "Given the following" in ex['prompt'])
        else:
            outputs = data.filter(lambda ex: "Given the following" not in ex['prompt'])
        scores[with_prompt] = {}
        for k in range(1,4):
            predictions = []
            references = []

            step_outputs = outputs.filter(lambda ex: len(ex['examples']) == k)
            
            for ex in step_outputs:
                generated_def = ex['generated definition'].split(".")[0]
                reference = true_defs[ex['word']]

                predictions.append(generated_def)
                references.append(reference)
            
            scores[with_prompt][k] = bertscore.compute(predictions=predictions, references=references, lang='en')
        
                